In [53]:
import pandas as pd

# display columns without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# display rows without truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.expand_frame_repr", False)

In [54]:
roles_only = pd.read_csv(r"D:\commo\code\id_role_assignment\assigned_only_roles.csv")
# standardized pillars
node_list1 = pd.read_csv(r"D:\commo\code\cleaned_pillars\standardized_pillar1.csv")
node_list2 = pd.read_csv(r"D:\commo\code\cleaned_pillars\standardized_pillar2.csv")
node_list3 = pd.read_csv(r"D:\commo\code\cleaned_pillars\standardized_pillar3.csv")

In [ ]:
"""
The node lists for kumu contain columns such as (Id), Label, Type, Description, (Tags), and other cols according to each entity type. 
In the edge list, it contains columns such as From, To, Direction, Type, Strength, and Description.
"""

In [55]:
# re order the columns in node_list1

node_list1.rename(columns={'Company Name': 'Label'}, inplace=True)
# move the label col to index 0
# Get the name of the label column at index 0
label_col = node_list1.columns[1]
# Pop the column and insert it at index 0, run one time
node_list1.insert(0, 'Label', node_list1.pop(label_col))

# insert the Type col, which is the role(s)
role_map = roles_only.set_index("name")["role(s)"]
node_list1.insert(1, "Type", pd.NA) # run one time
node_list1["Type"] = node_list1["Label"].map(role_map) 

# move the Notes to index 2 as description
desc_col = node_list1.columns[-1]
node_list1.insert(2, 'Description', node_list1.pop(desc_col))
# save node_list1


In [56]:
# re order the columns in node_list2

# rename the firm name as label
node_list2.rename(columns={'Firm Name': 'Label'}, inplace=True)

# insert the Type col, which is the role(s)
role_map = roles_only.set_index("name")["role(s)"]
node_list2.insert(1, "Type", pd.NA)  # run one time
node_list2["Type"] = node_list2["Label"].map(role_map)

# rename the Type (intl/local)
node_list2.rename(columns={'Type (Intl / Local)': 'Scope'}, inplace=True)

# move the notable cases + notes to description col at index 2
col1 = "Notable Cases / Disputes"
col2 = "Notes"
def build_description(row):
    val1 = row[col1] if isinstance(row[col1], str) and row[col1].strip() else "N/A"
    val2 = row[col2] if isinstance(row[col2], str) and row[col2].strip() else "N/A"
    return f"{col1}: {val1}\n\n{col2}: {val2}"
node_list2["Description"] = node_list2.apply(build_description, axis=1)
node_list2.insert(2, "Description", node_list2.pop("Description"))
# drop the last 2 columns
node_list2 = node_list2.drop(columns=["Notable Cases / Disputes", "Notes"])
# save node_list2

In [57]:
# re order the columns in node_list3
node_list3.rename(columns={'Institution Name': 'Label', 'Category':'Financier Type'}, inplace=True)

# insert the Type col, which is the role(s) at index 1
role_map = roles_only.set_index("name")["role(s)"]
node_list3.insert(1, "Type", pd.NA)  # run one time
node_list3["Type"] = node_list3["Label"].map(role_map)

# move the Notes to index 2 as description
node_list3.insert(2, "Description", node_list3.pop('Notes'))
# save node_list3

In [ ]:
# In the edge list, it contains columns such as From, To, Direction, Type
edge_list = pd.DataFrame(columns=['From', 'To', 'Direction', 'Type'])

# note the node_list2 Label (edge type: counsels_for) Known Client Base
# explode node_list2 known client base into individual rows
import re
def split_names(row):
    if not isinstance(row, str) or not row.strip(): # if not row.strip() = True > not real content
        return []
    parts = re.split(r',\s*(?![^()]*\))', row)
    return [p.strip() for p in parts]
node_list2_exploded = node_list2[['Label', 'Known Client Base']].copy()
node_list2_exploded['Known Client Base'] = node_list2_exploded['Known Client Base'].apply(split_names) # each client row has list of comma-separated str
node_list2_exploded = node_list2_exploded.explode('Known Client Base').reset_index(drop=True) # each row has one client but same law firm
# change Label to 'From', known client base to 'to'
node_list2_exploded.rename(columns={'Label': 'From', 'Known Client Base': 'To'}, inplace=True)
# add Direction column
node_list2_exploded['Direction'] = 'directed'
# add edge type column
node_list2_exploded['Type'] = 'counsels_for'
# node_list2_exploded is ready for concat

# note the node_list3 Label (edge type: lends_to) Known Clients
# explode node_list3 known clients into individual rows
node_list3_exploded = node_list3[['Label', 'Known Clients']].copy()
node_list3_exploded['Known Clients'] = node_list3_exploded['Known Clients'].apply(split_names)
node_list3_exploded = node_list3_exploded.explode('Known Clients').reset_index(drop=True) #before reset_index, the exploded rows still have their original row index
# rename Label to from, Known Clients to to
node_list3_exploded.rename(columns={'Label':'From', 'Known Clients':'To'}, inplace=True)
node_list3_exploded['Direction'] = 'directed'
node_list3_exploded['Type'] = 'lends_to'
#node_list3_exploded is ready for concat

# node_list1 lends_to? owns?
# trafigura group owns trafigura carbon trading, trafigura maritime logistics
# vitol group owns vitol asia
# mercuria energy group owns mercuria holdings
# gunvor group owns gunvor singapore
# olam group owns olam food ingredients, olam agri holdings
# louis dreyfus company owns louis dreyfus company asia
# cofco international owns cofco international singapore
# glencore owns glencore singapore
# mitsui & co owns mitsui & co energy trading singapore (mets)
# shell singapore owns shell eastern trading
# sinopec owns unipec singapore, sinopec international, sinopec international singapore, sinopec fuel oil
# petrochina international owns petrochina international singapore
# sinar mas group owns golden agri international
# standard chartered owns standard chartered bank singapore scb
# ed&f man capital owns ed&f man capital markets singapore
# wilmar international owns wilmar sugar
# cargill owns cargill asia pacific holdings
# bunge owns bunge asia
# trafigura group lends to lex oil, develop global, woodlawn mine holdings, tarago operations
# gunvor group lends to amaroq
# vitol group lends to uganda national oil unoc
# glencore lends to katanga mining
# mercuria energy group / mercuria energy trading lends to kipushi corporation sa, vast resources

,From,To,Direction,Type
0,ing,glencore,directed,lends_to
1,ing,gunvor group,directed,lends_to
2,ing,trafigura group,directed,lends_to
3,ing,vitol group,directed,lends_to
4,societe generale,NaN,directed,lends_to
5,bnp paribas,NaN,directed,lends_to
6,rabobank,NaN,directed,lends_to
7,standard chartered,glencore,directed,lends_to
8,standard chartered,marubeni corporation,directed,lends_to
9,standard chartered,seatrium,directed,lends_to


In [63]:
node_list3_exploded

,From,To,Direction,Type
0,ing,glencore,directed,lends_to
1,ing,gunvor group,directed,lends_to
2,ing,trafigura group,directed,lends_to
3,ing,vitol group,directed,lends_to
4,societe generale,NaN,directed,lends_to
5,bnp paribas,NaN,directed,lends_to
6,rabobank,NaN,directed,lends_to
7,standard chartered,glencore,directed,lends_to
8,standard chartered,marubeni corporation,directed,lends_to
9,standard chartered,seatrium,directed,lends_to
